# Select features to add MA with XGB

In [8]:
import os
import pickle
import polars as pl
import numpy as np
import pandas as pd
import torch
device = torch.device(f'cuda:2' if torch.cuda.is_available() else 'cpu')
accelerator = 'gpu' if torch.cuda.is_available() else 'cpu'

# Load Data

In [9]:
# df = pl.scan_parquet('train.parquet').collect().to_pandas()
# valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
# feature_names = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{idx}_lag_1" for idx in range(9)]
# label_name = 'responder_6'
# X_train = df[ feature_names ]
# y_train = df[ label_name ]
# w_train = df[ "weight" ]
# X_valid = valid[ feature_names ]
# y_valid = valid[ label_name ]
# w_valid = valid[ "weight" ]

# X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

## Select Features with MA window size 4

In [ ]:
# moving average
def add_moving_averages(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    new_columns = {}
    for col in columns_to_transform:
        new_col_name = f"{col}_ma{window}"
        new_columns[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).mean())
        )
    new_columns_df = pd.DataFrame(new_columns)
    df = pd.concat([df, new_columns_df], axis=1)
    return df

# now add function for sample var in sliding window
def add_sample_var(df, columns_to_transform, date_col='date_id', window=4, min_periods=1):
    new_columns = {}
    for col in columns_to_transform:
        new_col_name = f"{col}_var{window}"
        new_columns[new_col_name] = (
            df.groupby(date_col)[col]
              .transform(lambda x: x.rolling(window=window, min_periods=min_periods).var())
        )
    new_columns_df = pd.DataFrame(new_columns)
    df = pd.concat([df, new_columns_df], axis=1)
    return df


window_size = 4

features_for_ma = {}
features_for_ma[4] =  ["feature_10", "feature_42", "feature_04"]

features_for_var = {}
features_for_var[4] = ['feature_20', 'feature_21' ,'feature_15']

features_for_ma[7] = [f"feature_{i:02d}" for i in range(79)]
features_for_var[7] = [f"feature_{i:02d}" for i in range(79)]

for i in [4, 7]:
    df = add_moving_averages(df, features_for_ma[i], window=i)
    df = add_sample_var(df, features_for_var[i], window=i)
    valid = add_moving_averages(valid, features_for_ma[i], window=i)
    valid = add_sample_var(valid, features_for_var[i], window=i)


label_name = 'responder_6'
weight_name = 'weight'

In [11]:
use_ma_features = True
use_var_features = True
feature_names = [f"feature_{i:02d}" for i in range(79)] + \
                [f"responder_{idx}_lag_1" for idx in range(9)]

if use_ma_features:
    for window_size in features_for_ma:
        ma_feature_names = [f"{col}_ma{window_size}" for col in features_for_ma[window_size]]
        feature_names += ma_feature_names
if use_var_features:
    for window_size in features_for_var:
        var_feature_names = [f"{col}_var{window_size}" for col in features_for_var[window_size] ]
        feature_names += var_feature_names

feature_names[-10:], len(feature_names)

(['feature_69_var7',
  'feature_70_var7',
  'feature_71_var7',
  'feature_72_var7',
  'feature_73_var7',
  'feature_74_var7',
  'feature_75_var7',
  'feature_76_var7',
  'feature_77_var7',
  'feature_78_var7'],
 254)

In [12]:
# df.shape # (21022056, 270)
# save_parquet = False
# if save_parquet:
#     df.to_parquet('window7_all_mean_and_var.parquet', index=False)


# Train grand XGB on all 88 + 8 + 79*2 = 254 features, and find the most important ones

In [13]:
# import pandas as pd
# import numpy as np
# from xgboost import XGBRegressor

# # Extract data from the dataframe
# X = df[feature_names]
# y = df[label_name]
# if weight_name in df.columns:
#     w = df[weight_name]

# model = XGBRegressor(
#     tree_method='gpu_hist',    # Build trees on the GPU
#     device = "cuda",          # Use the GPU
#     predictor='gpu_predictor', # Do inference on the GPU
#     gpu_id=2, 
#     random_state=42,
#     n_estimators=100,
#     learning_rate=0.1,
#     max_depth=6,
#     objective="reg:squarederror"
# )
# print(X.shape, y.shape)
# print(X.dtypes)



# print('start training')
# model.fit(X, y, sample_weight=w)

# # -------------------------------------
# # Extract feature importances
# # -------------------------------------
# importances = model.feature_importances_

# importance_df = pd.DataFrame({
#     "feature": feature_names,
#     "importance": importances
# })

# # Sort features by importance descending
# importance_df.sort_values("importance", ascending=False, inplace=True)
# importance_df.reset_index(drop=True, inplace=True)

# importance_df




In [14]:
import pyarrow.parquet as pq
import xgboost as xgb
import pandas as pd

# ---------------------------------------------------------
# 1. Define the XGBoost parameters.
#    Make sure to use GPU-specific parameters if you want
#    to train on a GPU.
# ---------------------------------------------------------
params = {
    "tree_method": "gpu_hist",    # Build trees on the GPU
    "predictor": "gpu_predictor", # GPU-based predictions
    "gpu_id": 2,                  # use GPU index 2
    "random_state": 42,
    "objective": "reg:squarederror"
}
num_rounds_per_chunk = 10  # Number of boosting rounds per chunk (tune as needed)
file = pq.ParquetFile('window7_all_mean_and_var.parquet')

# We will store our trained booster here once we create it
booster = None

# ---------------------------------------------------------
# 3. Iterate row group by row group
# ---------------------------------------------------------
for i in range(file.num_row_groups):
    print(f"Reading row group {i+1}/{file.num_row_groups}")
    # Read this row group into a Pandas DataFrame
    row_group_df = file.read_row_group(i).to_pandas()

    # Separate features, label, and optionally weight
    X_chunk = row_group_df[feature_names]
    y_chunk = row_group_df[label_name]
    
    # If you have sample weights:
    if weight_name in row_group_df.columns:
        w_chunk = row_group_df[weight_name]
        dtrain = xgb.DMatrix(data=X_chunk, label=y_chunk, weight=w_chunk)
    else:
        dtrain = xgb.DMatrix(data=X_chunk, label=y_chunk)

    # -----------------------------------------------------
    # 4. Train incrementally with xgb.train()
    #
    # - If booster is None (the first chunk), train from scratch.
    # - Otherwise, pass the previously trained booster in xgb_model=booster
    #   so we continue from where we left off.
    # -----------------------------------------------------
    if booster is None:
        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=num_rounds_per_chunk,
        )
    else:
        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=num_rounds_per_chunk,
            xgb_model=booster  # warm-start from previous state
        )
    print(f"Finished training on row group {i+1}.")

# ---------------------------------------------------------
# 5. Once training is complete, you have your final booster.
#    You can get feature importances or save the model.
# ---------------------------------------------------------

# Example: Extract raw importance dictionary
importance_dict = booster.get_score(importance_type="gain")

# Convert it into a sorted DataFrame
importance_df = (
    pd.DataFrame.from_dict(importance_dict, orient='index', columns=['importance'])
    .reset_index()
    .rename(columns={'index': 'feature'})
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
)

print(importance_df.head(20))


Reading row group 1/21


/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:06:21] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:06:21] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:06:21] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


Finished training on row group 1.
Reading row group 2/21


/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:06:24] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:06:24] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


Finished training on row group 2.
Reading row group 3/21


...

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


Finished training on row group 19.
Reading row group 20/21


/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:09] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:09] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


Finished training on row group 20.
Reading row group 21/21
Finished training on row group 21.
              feature   importance
0          feature_23  6993.765137
1     feature_10_var7  4942.292969
2     feature_20_var7  3899.435547
3      feature_28_ma7  3778.894775
4      feature_10_ma7  3448.472168
5          feature_28  3281.547607
6      feature_24_ma7  3184.080322
7      feature_10_ma4  3147.676758
8          feature_09  3115.534424
9     feature_24_var7  3113.226562
10     feature_21_ma7  3100.340332
11     feature_22_ma7  3086.633789
12    feature_10_var4  3076.466797
13    feature_23_var7  2981.343018
14    feature_20_var4  2944.099121
15  responder_7_lag_1  2878.672852
16     feature_30_ma7  2793.759521
17         feature_25  2752.713623
18  responder_3_lag_1  2666.124756
19     feature_23_ma7  2657.569580


/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:10] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/xgboost/core.py:158: UserWarning: [18:07:10] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


In [19]:
score_dict = booster.get_score(importance_type="gain")
# Sum all gains
total_gain = sum(score_dict.values())
# Convert to fraction of total
normalized = {k: v / total_gain for k, v in score_dict.items()}
normalized_df = (
    pd.DataFrame.from_dict(normalized, orient='index', columns=['importance'])
    .reset_index()
    .rename(columns={'index': 'feature'})
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
)
normalized_df.head(20)

,feature,importance
0,feature_23,0.028617
1,feature_10_var7,0.020223
2,feature_20_var7,0.015956
3,feature_28_ma7,0.015463
4,feature_10_ma7,0.014110
5,feature_28,0.013427
6,feature_24_ma7,0.013029
7,feature_10_ma4,0.012880
8,feature_09,0.012748
9,feature_24_var7,0.012739


In [16]:
ma_feature_names = importance_df[:50]["feature"].tolist()
top_ma_features = [col for col in ma_feature_names if col.endswith(f"ma{window_size}")]
top_var_features = [col for col in importance_df[:50]["feature"].tolist() if col.endswith(f"var{window_size}")]
top_ma_features, top_var_features

(['feature_28_ma7',
  'feature_10_ma7',
  'feature_24_ma7',
  'feature_21_ma7',
  'feature_22_ma7',
  'feature_30_ma7',
  'feature_23_ma7',
  'feature_25_ma7',
  'feature_29_ma7',
  'feature_20_ma7',
  'feature_27_ma7',
  'feature_31_ma7',
  'feature_11_ma7',
  'feature_26_ma7',
  'feature_02_ma7'],
 ['feature_10_var7',
  'feature_20_var7',
  'feature_24_var7',
  'feature_23_var7',
  'feature_30_var7',
  'feature_26_var7',
  'feature_27_var7',
  'feature_28_var7',
  'feature_21_var7',
  'feature_29_var7',
  'feature_31_var7'])

## Add Features
